# BERT fine-tuning

This notebook fine-tunes `bert-base-uncased` on our training data. It is the transformer
baseline the professor named, and it is the first model in the benchmark where the whole
network is trained rather than used as a fixed feature extractor.

**Architecture vs specific model.** BERT is the architecture: a 12-layer transformer encoder
trained with masked language modelling. `bert-base-uncased` is the specific checkpoint we
load: about 110M parameters, English only, lowercased input.

**What fine-tuning means here.** In notebook 03 the sentence transformer was *frozen* — it
only produced vectors, and the only thing trained was the SVM on top. Here every weight in
BERT is updated by our 2,104 training messages, plus a new 14-way classification head.
That is the difference the benchmark is trying to measure.

Both test sets are scored once at the end: the synthetic test set and the real MINDS-14
messages. The score to beat on MINDS-14 is **0.951** (Jev).
## 0. Colab setup

This notebook needs a GPU. In Colab: **Runtime → Change runtime type → T4 GPU**.

The cell below clones the repository. The token is typed at runtime with `getpass`, so it is
never stored in the notebook file and never appears in the output. Skip this cell if you are
running locally from `notebooks/`.

In [1]:
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    import getpass, os, subprocess

    REPO = "IbragimAliyev/Intent_Classification_For_Banking"
    token = getpass.getpass("GitHub token (input hidden): ")
    result = subprocess.run(
        ["git", "clone", f"https://{token}@github.com/{REPO}.git", "/content/repo"],
        capture_output=True, text=True,
    )
    print("clone ok" if result.returncode == 0
          else result.stderr.replace(token, "***"))
    del token

    os.chdir("/content/repo/notebooks")
    !pip -q install "transformers>=4.46" "accelerate>=0.26"

print("working dir:", __import__("os").getcwd())

GitHub token (input hidden): ··········
clone ok
working dir: /content/repo/notebooks


## 1. Imports

`src/finetune.py` holds the training code. It is shared with notebook 06 (XLM-RoBERTa) so
both models are trained by exactly the same function — same seed, same batch size, same
early-stopping rule. If the two were trained by separate code, a difference between them
could come from the training setup rather than from the model.

In [2]:
import os
import pandas as pd
import torch

import sys
sys.path.insert(0, '..')


from src.evaluate import evaluate
from src.finetune import finetune, predict_intents, best_epoch_score
SEED = 42
MODEL_NAME = "bert-base-uncased"

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — stop and switch runtime to T4")

GPU: Tesla T4


## 2. Load the data

The same split as every other notebook, plus the real MINDS-14 messages Mahmut prepared.
MINDS-14 is a test set only — we never train on it.
- **train**: fine-tunes the model
- **validation**: picks the learning rate and the epoch
- **synthetic test** and **minds14**: scored once each, at the very end

In [3]:
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")
minds_df = pd.read_csv("../data/processed/minds14_en.csv")

print("train:  ", train_df.shape)
print("val:    ", val_df.shape)
print("test:   ", test_df.shape)
print("minds14:", minds_df.shape)
print()
print("intents match:", sorted(train_df['intent'].unique()) == sorted(minds_df['intent'].unique()))

train:   (2104, 2)
val:     (451, 2)
test:    (452, 2)
minds14: (1809, 3)

intents match: True


## 3. Check the message lengths

`max_length` decides where BERT cuts a message off. Too small and we lose words; too large
and we waste memory and time padding. We check the real lengths before fixing the value.

In [4]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
for name, frame in [("train", train_df), ("synthetic test", test_df), ("minds14", minds_df)]:
    lengths = [len(tok.encode(str(t))) for t in frame["text"]]
    over = sum(1 for n in lengths if n > 64)
    print(f"{name:15} max {max(lengths):3d}   mean {sum(lengths)/len(lengths):5.1f}   over 64: {over} ({100*over/len(lengths):.1f}%)")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

train           max  52   mean  17.8   over 64: 0 (0.0%)
synthetic test  max  54   mean  17.4   over 64: 0 (0.0%)
minds14         max  84   mean  19.7   over 64: 18 (1.0%)


## 4. Tune the learning rate on validation

Learning rate is the setting that matters most when fine-tuning a transformer. Too high and
the pretrained weights get destroyed in the first few steps; too low and the model barely
moves. `2e-5`, `3e-5` and `5e-5` are the standard range for BERT.

Each run trains up to 5 epochs and evaluates on validation after every epoch. The epoch with
the best validation macro-F1 is kept (`load_best_model_at_end`), so a run that starts to
overfit in later epochs is not penalised for it — we simply keep its best point.

The test sets are not touched anywhere in this section.

In [5]:
learning_rates = [2e-5, 3e-5, 5e-5]
results = []
trainers = {}

for lr in learning_rates:
    print(f"\n=== learning rate {lr} ===")
    trainer, labels = finetune(
        MODEL_NAME, train_df, val_df,
        learning_rate=lr, num_epochs=5, batch_size=16, seed=SEED,
    )
    epoch, score = best_epoch_score(trainer)
    results.append({"learning_rate": lr, "best_epoch": epoch, "val_macro_f1": round(score, 4)})
    trainers[lr] = trainer

tuning_df = pd.DataFrame(results)
print()
print(tuning_df)

best_lr = tuning_df.loc[tuning_df["val_macro_f1"].idxmax(), "learning_rate"]
print("\nBest learning rate:", best_lr)


=== learning rate 2e-05 ===


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte


   learning_rate  best_epoch  val_macro_f1
0        0.00002           5        0.9736
1        0.00003           4        0.9867
2        0.00005           4        0.9845

Best learning rate: 3e-05


## 5. The final model

The run with the best validation macro-F1 already has its best epoch loaded back in, so that
trainer *is* the final model. Retraining it with the same learning rate and the same seed
would reproduce it exactly, so we reuse it instead of spending another five epochs of GPU
time on an identical result.

In [6]:
final_trainer = trainers[best_lr]
best_row = tuning_df.loc[tuning_df["val_macro_f1"].idxmax()]
print(f"learning rate {best_row['learning_rate']}, epoch {best_row['best_epoch']}, "
      f"validation macro-F1 {best_row['val_macro_f1']}")

learning rate 3e-05, epoch 4.0, validation macro-F1 0.9867


## 6. Evaluate on both test sets, once

Scored with the shared `evaluate()` so the rows land in `results/metrics.csv` next to every
other model. **Run this cell once.** Each run appends a row; a second run creates duplicates
that somebody has to delete by hand.

In [7]:
test_preds = predict_intents(final_trainer, test_df["text"], labels, max_length=128)
minds_preds = predict_intents(final_trainer, minds_df["text"], labels, max_length=128)

evaluate(test_df["intent"], test_preds, "bert_base", "synthetic_test", out_dir="../results")
evaluate(minds_df["intent"], minds_preds, "bert_base", "minds14", out_dir="../results")

=== bert_base on synthetic_test ===
Accuracy: 0.9801   Macro-F1: 0.9799

                     precision    recall  f1-score   support

             abroad       1.00      1.00      1.00        32
            address       1.00      1.00      1.00        32
          app_error       1.00      1.00      1.00        33
          atm_limit       0.97      1.00      0.98        32
            balance       0.97      0.94      0.95        32
      business_loan       1.00      1.00      1.00        33
        card_issues       0.94      0.94      0.94        32
       cash_deposit       1.00      0.94      0.97        33
       direct_debit       0.94      1.00      0.97        32
             freeze       0.97      0.94      0.95        32
 high_value_payment       1.00      0.97      0.98        32
      joint_account       1.00      1.00      1.00        32
latest_transactions       0.94      1.00      0.97        32
           pay_bill       1.00      1.00      1.00        33

          

{'model': 'bert_base',
 'dataset': 'minds14',
 'accuracy': 0.9149,
 'macro_f1': 0.9155}

## 7. Save the predictions

`models/` is not in git and the fine-tuned weights live on Colab, so the predictions are the
only thing that survives. Mahmut's bootstrap confidence intervals and the final benchmark
table are both built from these files, not from the model.

In [8]:
os.makedirs("../results/predictions", exist_ok=True)

pd.DataFrame({"text": test_df["text"], "intent": test_df["intent"], "pred": test_preds}).to_csv(
    "../results/predictions/bert_base_synthetic_test.csv", index=False
)
pd.DataFrame({"text": minds_df["text"], "intent": minds_df["intent"], "pred": minds_preds}).to_csv(
    "../results/predictions/bert_base_minds14.csv", index=False
)
print("saved both prediction files")

saved both prediction files


## 8. Compare with the models already in the benchmark

The numbers below come straight from `results/metrics.csv`, so they include the rows this
notebook just added.

In [9]:
metrics = pd.read_csv("../results/metrics.csv")
for dataset in ["synthetic_test", "minds14"]:
    print(f"\n=== {dataset} ===")
    print(metrics[metrics["dataset"] == dataset]
          .sort_values("macro_f1", ascending=False)
          .to_string(index=False))


=== synthetic_test ===
    model        dataset  accuracy  macro_f1
      jev synthetic_test    0.9889    0.9890
sbert_svm synthetic_test    0.9867    0.9866
bert_base synthetic_test    0.9801    0.9799
 tfidf_lr synthetic_test    0.9403    0.9386

=== minds14 ===
    model dataset  accuracy  macro_f1
      jev minds14    0.9508    0.9510
sbert_svm minds14    0.9193    0.9170
bert_base minds14    0.9149    0.9155
 tfidf_lr minds14    0.8657    0.8640


## 9. Conclusion

- Best learning rate: 3e-05 , best epoch: 4
- Validation macro-F1: 0.9867
- synthetic_test: accuracy 0.9801 | macro-F1 0.9799
- minds14: accuracy 0.9149 | macro-F1 0.9155 (the score to beat was 0.951 from jev so we did not beat it)
- Fine-tuned BERT vs the frozen encoder in notebook 03 (sbert_svm: 0.9866 synthetic, 0.9170 minds14): bert_base came out slightly lower on both (0.9799 and 0.9155). Something to note is that the minds14 gap is only 0.0015 which is around 3 messages out of 1809 , so the two should be treated as tied until the bootstrap intervals say otherwise. Fine-tuning 110M parameters on only 2104 messages is not enough to beat a sentence encoder that was already pretrained on about a billion sentence pairs for this exact kind of short text similarity
- Drop from synthetic to real data: bert_base drops 6.4 points , sbert_svm 7.0 , tfidf_lr 7.5 and jev only 3.8 . The main takeaway is that the harder a model trains on our synthetic data the worse it transfers to real speech , jev never saw our data at all and it transfers the best , thus we can say the synthetic training set is the limit here and not the model
- Weakest intents on minds14 (from the confusion matrix): high_value_payment (F1 0.79 with precision 0.72 so it is over predicting this intent) , pay_bill (recall 0.80) and app_error (recall 0.82)